In [0]:
df = spark.table("workspace.default.bronze_hospital_raw")

from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])
null_counts.display()

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,0,2273,0,0,98569,0,0,0,0,40256,49949,0,0,0,0,0,0,21,358,1423,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import col, when, regexp_replace

df_silver = df \
    .drop("weight", "payer_code", "medical_specialty") \
    .dropDuplicates(["encounter_id"]) \
    .filter(col("discharge_disposition_id").isin([11, 13, 14, 19, 20, 21]) == False) \
    .withColumn("age_numeric", 
        regexp_replace(regexp_replace(col("age"), "\\[", ""), "\\-.*", "").cast("int")) \
    .withColumn("readmitted_binary",
        when(col("readmitted") == "<30", 1)
        .when(col("readmitted") == ">30", 1)
        .otherwise(0)) \
    .withColumn("gender", 
        when(col("gender") == "Unknown/Invalid", None).otherwise(col("gender"))) \
    .na.drop(subset=["race", "gender", "age"])

print(f"Silver rows: {df_silver.count()}")
print(f"Silver columns: {len(df_silver.columns)}")

Silver rows: 97108
Silver columns: 49


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_hospital_clean")

print("Silver table saved successfully")

Silver table saved successfully


In [0]:
spark.sql("""
    SELECT readmitted_binary, COUNT(*) as count 
    FROM workspace.default.silver_hospital_clean 
    GROUP BY readmitted_binary
""").display()

readmitted_binary,count
1,46090
0,51018
